# Nightly Garmin envelope–LIDS

Nightly inactivity profiles, cosine parameters, and an equal-night mean over four internal cycles. These inputs are **already cropped to sleep_period_time**.

This is an explicit Garmin adaptation: activity is the **sum of ten one-minute mean envelopes in mg**, not ZCM counts. Larger LIDS means less movement. See [LIDS.md](LIDS.md) for the complete method and reference links.

In [ ]:
from pathlib import Path
import os
import json
import sys

import matplotlib.pyplot as plt
from IPython.display import display

MODULE_DIR = Path.cwd() if (Path.cwd() / "lids_analysis.py").exists() else Path.cwd() / "offline_processing"
if str(MODULE_DIR) not in sys.path:
    sys.path.insert(0, str(MODULE_DIR))
from lids_analysis import LIDSConfig, analyze_nights, plot_night, plot_average, save_results

# Optional private paths in lids.local.json; environment overrides the file.
local_config = MODULE_DIR / "lids.local.json"
local_paths = json.loads(local_config.read_text()) if local_config.exists() else {}
PROCESSED_ROOT = Path(os.environ.get("OVERNIGHT_PROCESSED_ROOT", local_paths.get("processed_root", "data/garmin_processed")))
if not PROCESSED_ROOT.is_absolute():
    PROCESSED_ROOT = MODULE_DIR / PROCESSED_ROOT
OUTPUT_DIR = MODULE_DIR / "outputs" / "lids_envelope"
CONFIG = LIDSConfig(
    timezone="Europe/Rome", input_unit="mg", sampling_rate=100,
    min_coverage=0.90, fit_method="projection",
)
# Optional: fit_method="least_squares" jointly fits cosine, sine and intercept.
print(f"Input: {PROCESSED_ROOT.resolve()}")
print(f"Output: {OUTPUT_DIR.resolve()}")

## Minute envelope means → 10-minute sums → LIDS

Use the same burst-analysis signal: acceleration magnitude, 0.1–10 Hz order-8 zero-phase band-pass, upper minus lower envelope (groups of ten extrema). Convert from g to mg and **average each minute**, then **sum ten valid minute means** and apply `LIDS = 100 / (1 + activity_sum_mg)`. Finally use a centered 30-minute mean.

Minutes need 90% coverage; all ten minutes are required for a sum. Invalid samples and acquisition gaps split filtering runs. No burst threshold is applied. The mg scaling is explicit and affects LIDS values; this is not calibrated ZCM.

The default projection fit follows the referenced LIDS analysis: scan 30–180 minutes in 5-minute steps, maximizing `MRI = 2 × amplitude × r`. Phase uses `offset + amplitude × cos(2πt / period − phase)`. R² is `1 − SSE/SST` and can be negative for projection fits. All fit metrics are descriptive.

In [ ]:
result = analyze_nights(PROCESSED_ROOT, CONFIG, progress=print)
summary = result["summary"]
columns = ["status", "period_min", "amplitude", "phase_deg", "peak_time_min",
           "pearson_r", "r2", "rmse", "mri", "period_at_boundary", "bins_valid", "bins_total"]
display(summary.reindex(columns=columns).round(3))
failed = summary.loc[summary.status != "ok"]
if not failed.empty:
    display(failed)
print(f"Successful fits: {(summary.status == 'ok').sum()} / {len(summary)}")

# Inspect the minute means that feed each ten-minute sum.
if result["minutes"]:
    first_night = next(iter(result["minutes"]))
    display(result["minutes"][first_night].head(15))

## Each night

The pale curve shows the unsmoothed 10-minute LIDS, the teal curve the 30-minute mean, and the dashed curve the fitted cosine. Dates are recording-folder identifiers; the x axis shows local time. Fit parameters are also exported as a table.

In [ ]:
for night, bins in result["bouts"].items():
    fig = plot_night(night, bins, summary.loc[night])
    plt.show()
    plt.close(fig)

## Mean profile in internal time

For each fitted bout: `internal_min = external_min / fitted_period_min × 90`.
Interpolate the observed smoothed LIDS at 10-minute positions over four cycles (`0…350` minutes). No phase alignment, amplitude normalization, extrapolation, or gap filling. Average the available nights equally in each bin.

Shading is mean ± SEM across nights. The lower panel shows actual contributors, and the table gives retention by cycle. Around 50% retention at cycle four is not assumed; these data determine the counts.

In [ ]:
fig = plot_average(result)
plt.show()
plt.close(fig)
display(result["cycles"])
display(result["profile"].round(3))

## Save local results

Writes nightly PNGs, one-minute mean envelopes with coverage, and 10-minute binned data, summary parameters, rescaled individual profiles, mean/SEM/counts, per-cycle retention, settings, and the average PNG/SVG. This directory is Git-ignored. Keep an executed notebook copy here if you want to preserve its output; publish only the clean source notebook.

New results go to `outputs/lids_envelope/`. Previous ENMO outputs stay in `outputs/lids/`.

In [ ]:
save_results(result, OUTPUT_DIR)
print(f"Saved tables and plots to {OUTPUT_DIR.resolve()}")